# 02 — QLoRA Fine-Tuning Gemma 2 2B on Medical QA

Fine-tune Gemma 2 2B using QLoRA (4-bit base + LoRA adapters) via **standard PEFT + BitsAndBytes**
on combined PubMedQA + MedQA data. Then merge adapters back into the base model.

**Run this on**: Kaggle (T4 16GB) — Gemma 2 2B fits easily on a single T4.

In [ ]:
# Install dependencies
!pip install -q "transformers>=5.5.0" peft bitsandbytes datasets accelerate sentencepiece protobuf trl pandas tqdm

import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
print(f"GPU 0: {torch.cuda.get_device_name(0)}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"VRAM GPU0: {torch.cuda.mem_get_info(0)[1] / 1024**3:.1f} GB")
if torch.cuda.device_count() > 1:
    print(f"VRAM GPU1: {torch.cuda.mem_get_info(1)[1] / 1024**3:.1f} GB")

## 1. Load Model in 4-bit with PEFT + BitsAndBytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_NAME = "google/gemma-2-2b"
MAX_SEQ_LENGTH = 512

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading {MODEL_NAME} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded. VRAM: {torch.cuda.memory_allocated()/1024**3:.1f}GB")

## 2. Add LoRA Adapters

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

## 3. Load Training Data

In [ ]:
from datasets import load_dataset, concatenate_datasets

# Gemma 2 prompt templates
PUBMEDQA_TEMPLATE = """<start_of_turn>user
Context: {context}

Question: {question}
Answer with yes, no, or maybe and explain your reasoning.<end_of_turn>
<start_of_turn>model
{answer}<end_of_turn>"""

MEDQA_TEMPLATE = """<start_of_turn>user
{question}<end_of_turn>
<start_of_turn>model
{answer}<end_of_turn>"""


def format_pubmedqa(example):
    ctx_data = example.get("context", {})
    contexts = ctx_data.get("contexts", [])
    context_str = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
    long_answer = example.get("long_answer", "")
    final_decision = example.get("final_decision", "")
    answer = f"{final_decision}. {long_answer}" if long_answer else final_decision
    return {"text": PUBMEDQA_TEMPLATE.format(context=context_str, question=example["question"], answer=answer)}


def format_medqa(example):
    question = example["question"]
    options = example.get("options", {})
    if isinstance(options, dict):
        opts_str = "\n".join(f"  {k}) {v}" for k, v in sorted(options.items()))
        question = f"{question}\n{opts_str}"
    answer_idx = example.get("answer_idx", "")
    answer = example.get("answer", "")
    full_answer = f"The answer is {answer_idx}) {answer}" if answer_idx else answer
    return {"text": MEDQA_TEMPLATE.format(question=question, answer=full_answer)}


pubmedqa = load_dataset("pubmed_qa", "pqa_labeled")
medqa = load_dataset("GBaker/MedQA-USMLE-4-options")

pubmedqa_fmt = pubmedqa.map(format_pubmedqa, remove_columns=pubmedqa["train"].column_names)
medqa_fmt = medqa.map(format_medqa, remove_columns=medqa["train"].column_names)

train_dataset = concatenate_datasets([pubmedqa_fmt["train"], medqa_fmt["train"]])
val_dataset = medqa_fmt["test"]

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

## 4. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "gemma2-2b-medical-qlora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,       # Effective batch = 16
    learning_rate=2e-4,
    warmup_steps=50,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
    optim="adamw_8bit",
    max_grad_norm=0.3,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Save the LoRA adapter
import gc
torch.cuda.empty_cache()
gc.collect()

trainer.save_model(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

# Log training metrics
train_log = trainer.state.log_history
print(f"\nFinal train loss: {train_log[-2].get('loss', 'N/A')}")
print(f"Final eval loss:  {train_log[-1].get('eval_loss', 'N/A')}")

## 5. Merge LoRA into Base Model

Merge the adapter weights back into the base model and save.
On 2xT4 we merge in-memory since the model is already split across GPUs.

In [ ]:
# Free training memory, keep model
del trainer
torch.cuda.empty_cache()
gc.collect()

MERGED_DIR = "gemma2-2b-medical-merged"

# Merge LoRA weights into the base model
print("Merging LoRA weights...")
merged_model = model.merge_and_unload()

print(f"Saving merged model to {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR, max_shard_size="2GB")
tokenizer.save_pretrained(MERGED_DIR)

print(f"Done! Merged model saved to {MERGED_DIR}/")

del merged_model, model
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Quick sanity check — load merged model in 4-bit and test generation
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR, quantization_config=bnb_config, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)

prompt = "<start_of_turn>user\nWhat are the common symptoms of Type 2 diabetes?<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# Evaluate the fine-tuned model
import time
import pandas as pd
from tqdm import tqdm
import gc

device = "cuda"
ft_results = {"model": "gemma2-2b-medical-finetuned"}
BATCH_SIZE = 4

print("[1/4] WikiText-2 perplexity...")
wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
wiki_texts = [t for t in wiki["text"] if len(t.strip()) > 50][:256]
model.eval()
total_loss, total_tokens = 0.0, 0
for i in tqdm(range(0, len(wiki_texts), BATCH_SIZE)):
    batch = wiki_texts[i:i+BATCH_SIZE]
    enc = tokenizer(batch, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        out = model(**enc, labels=enc["input_ids"])
    n = enc["attention_mask"].sum().item()
    total_loss += out.loss.item() * n
    total_tokens += n
    del enc, out
torch.cuda.empty_cache()
ft_results["perplexity_wikitext"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
print(f"  -> {ft_results['perplexity_wikitext']:.2f}")

print("[2/4] Medical text perplexity...")
pqa_eval = load_dataset("pubmed_qa", "pqa_labeled", split="train")
med_texts = []
for ex in pqa_eval:
    ctx_data = ex.get("context", {})
    contexts = ctx_data.get("contexts", [])
    ctx = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
    if len(ctx.strip()) > 50:
        med_texts.append(ctx)
    if len(med_texts) >= 256:
        break
total_loss, total_tokens = 0.0, 0
for i in tqdm(range(0, len(med_texts), BATCH_SIZE)):
    batch = med_texts[i:i+BATCH_SIZE]
    enc = tokenizer(batch, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        out = model(**enc, labels=enc["input_ids"])
    n = enc["attention_mask"].sum().item()
    total_loss += out.loss.item() * n
    total_tokens += n
    del enc, out
torch.cuda.empty_cache()
ft_results["perplexity_medical"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
print(f"  -> {ft_results['perplexity_medical']:.2f}")

print("[3/4] PubMedQA accuracy...")
correct, total = 0, 0
for ex in tqdm(pqa_eval, total=min(len(pqa_eval), 200)):
    if total >= 200:
        break
    ctx_data = ex.get("context", {})
    contexts = ctx_data.get("contexts", [])
    context = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
    prompt = (
        f"<start_of_turn>user\nContext: {context}\n\n"
        f"Question: {ex['question']}\n"
        f"Answer with exactly one word: yes, no, or maybe.<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    pred = resp.split()[0].strip(".,!?;:") if resp.split() else ""
    gold = ex["final_decision"].lower().strip()
    if pred in ("yes", "no", "maybe") and pred == gold:
        correct += 1
    total += 1
    del inputs, out
    torch.cuda.empty_cache()
ft_results["pubmedqa_accuracy"] = correct / total if total else 0
print(f"  -> {ft_results['pubmedqa_accuracy']:.4f} ({correct}/{total})")

print("[4/4] Inference speed & memory...")
prompt = "<start_of_turn>user\nWhat is diabetes?<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    model.generate(**inputs, max_new_tokens=50, do_sample=False)
total_tok, total_t = 0, 0.0
for _ in range(5):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=50, do_sample=False)
    torch.cuda.synchronize()
    total_tok += out.shape[1] - inputs["input_ids"].shape[1]
    total_t += time.perf_counter() - t0
ft_results["tokens_per_sec"] = total_tok / total_t
ft_results["peak_vram_gb"] = torch.cuda.max_memory_allocated() / (1024**3)
print(f"  -> {ft_results['tokens_per_sec']:.1f} tok/s, {ft_results['peak_vram_gb']:.2f} GB VRAM")

import os
os.makedirs("results/tables", exist_ok=True)
df_new = pd.DataFrame([ft_results])
if os.path.exists("results/tables/all_results.csv"):
    df = pd.read_csv("results/tables/all_results.csv")
    df = df[df["model"] != ft_results["model"]]
    df = pd.concat([df, df_new], ignore_index=True)
else:
    df = df_new
df.to_csv("results/tables/all_results.csv", index=False)
print("\nFine-tuned results saved.")
df